# 02 · Feature Engineering & Exploration
## Global CO2 Insight — OWID Dataset (79 columns → 19 core features)

**Dataset:** `data/processed/ml_ready.parquet`
**Source:** Our World in Data (OWID) CO2 dataset, preprocessed via `co2_ml.pipelines.preprocess`

### Contents
1. Data loading & overview
2. Top 10 emitter trends (1960–2023)
3. Feature correlation heatmap
4. Per-capita vs total emissions scatter

> **Note:** Full statistical tests (ADF stationarity, ACF/PACF) to be added in Phase 2 prep.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

df = pd.read_parquet("../data/processed/ml_ready.parquet")
print(f"Shape: {df.shape}")
print(f"Year range: {df['year'].min()}–{df['year'].max()}")
print(f"Countries: {df['iso_code'].nunique()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print("\nHead:")
df.head()

## 1. Top 10 Emitters — CO2 Trend (1960–2023)

In [ ]:
# Identify top 10 emitters by latest year CO2
latest_year = df["year"].max()
top10 = df[df["year"] == latest_year].nlargest(10, "co2")["country"].tolist()

fig, ax = plt.subplots(figsize=(14, 7))
for country in top10:
    subset = df[df["country"] == country].sort_values("year")
    ax.plot(subset["year"], subset["co2"], label=country, linewidth=2)

ax.set_title(f"Top 10 CO2 Emitters — Annual Emissions (1960–{latest_year})", fontsize=16)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("CO2 Emissions (million tonnes)", fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)
plt.tight_layout()
plt.savefig("../assets/eda_top10_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Top 10 emitters ({latest_year}): {top10}")

## 2. Feature Correlation Heatmap

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "year"]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Feature Correlation Matrix (OWID CO2 — 19 Core Features)", fontsize=16)
plt.tight_layout()
plt.savefig("../assets/eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Per-Capita vs Total Emissions (Latest Year)

In [ ]:
# Per-capita vs total CO2 scatter — latest year
df_latest = df[df["year"] == latest_year].dropna(subset=["co2", "co2_per_capita", "population"])

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(
    df_latest["co2"],
    df_latest["co2_per_capita"],
    s=df_latest["population"] / 1e6,  # bubble size = population
    alpha=0.5,
    edgecolors="black",
    linewidth=0.5,
)

# Label top emitters
for _, row in df_latest.nlargest(10, "co2").iterrows():
    ax.annotate(
        row["country"],
        (row["co2"], row["co2_per_capita"]),
        fontsize=9,
        fontweight="bold",
        xytext=(5, 5),
        textcoords="offset points",
    )

# Label high per-capita outliers
for _, row in df_latest.nlargest(5, "co2_per_capita").iterrows():
    if row["country"] not in df_latest.nlargest(10, "co2")["country"].values:
        ax.annotate(
            row["country"],
            (row["co2"], row["co2_per_capita"]),
            fontsize=9,
            fontweight="bold",
            color="red",
            xytext=(5, 5),
            textcoords="offset points",
        )

ax.set_xlabel("Total CO2 Emissions (million tonnes)", fontsize=12)
ax.set_ylabel("CO2 Per Capita (tonnes/person)", fontsize=12)
ax.set_title(f"Total vs Per-Capita CO2 Emissions ({latest_year}) — bubble size = population", fontsize=14)
ax.set_xscale("log")
plt.tight_layout()
plt.savefig("../assets/eda_percapita_scatter.png", dpi=150, bbox_inches="tight")
plt.show()